# Virtualize NASA Earthdata with earthaccess + VirtualiZarr

This notebook demonstrates the full pipeline for creating virtual (cloud-optimized) datasets from real NASA Earthdata collections without downloading science data.

**Pipeline overview**

1. **Search** — query NASA CMR for granules matching a collection, temporal range, and bounding box.
2. **Virtualize** — use `earthaccess.virtualize()` to open granules as virtual `ManifestArray`-backed datasets, with optional spatial mosaicking (`mosaic_dims`) and ragged-scanline padding (`pad="auto"`).
3. **Serialize** — write the result to a persistent virtual store (kerchunk JSON/parquet or icechunk).
4. **Compute** — open the store lazily and run analytics (e.g., `mean()`) with chunk-level cloud access.

**Supported workflows**

| Collection | Pattern | Key params |
|---|---|---|
| TEMPO L2 (NO₂, HCHO, etc.) | Ragged scanlines per orbit | `pad="auto"`, `concat_dim="time"` |
| ITS-LIVE | Spatially tiled velocity maps | `mosaic_dims=["y","x"]`, `pad="none"` |
| Any single-group NetCDF/HDF5 | Simple concatenation | `concat_dim` only |

**Dask integration**

By default `earthaccess.virtualize()` uses `parallel="dask"`, which wraps per-granule opens in `dask.delayed`.  To run on a **Dask Distributed** cluster, create a `Client` *before* calling `virtualize()` — the delayed tasks will then execute on the cluster workers.


## 1. Authentication

`earthaccess.login()` tries all available methods in order:
1. `EARTHDATA_USERNAME` / `EARTHDATA_PASSWORD` environment variables
2. `~/.netrc` file
3. Interactive browser / prompt

For unattended runs (e.g. on a cluster), export the env vars first.

In [1]:
import earthaccess

earthaccess.login()
earthaccess.__version__

'0.8.3.dev1234+g5bb8909e2'

## 2. Search granules

Query the NASA Common Metadata Repository (CMR) for granules.  Always check that the result set is homogeneous — same product version, same spatial grid, consistent variables.  Mixed versions will produce concatenation errors.

In [ ]:
import re

COLLECTION = "TEMPO_NO2_L2"
TEMPORAL = ("2024-03-28", "2024-03-29")
VERSION = "V03"          # filter to one product version
MAX_GRANULES = None        # set to e.g. 10 for a quick smoke test

granules = earthaccess.search_data(
    short_name=COLLECTION,
    temporal=TEMPORAL,
    version=VERSION
)


print(f"Found {len(granules)} granules")

if MAX_GRANULES:
    granules = granules[:MAX_GRANULES]
    print(f"Limited to first {MAX_GRANULES} granules")

## 3. (Optional) Start a Dask Distributed cluster

`earthaccess.virtualize(parallel="dask")` wraps each granule open in `dask.delayed`.  If a `distributed.Client` exists, those tasks run on the cluster.

**Local machine** — good for testing:
```python
from dask.distributed import Client
client = Client(n_workers=4, threads_per_worker=2)
```

**HPC / SLURM** — replace with your cluster-appropriate launcher (e.g. `dask_jobqueue.SLURMCluster`, `dask_gateway.GatewayCluster`, Coiled, etc.).


In [ ]:
# Uncomment to start a local Dask cluster.
# from dask.distributed import Client
# client = Client(n_workers=4, threads_per_worker=2)
# print(client.dashboard_link)

## 4. Virtualize

`earthaccess.virtualize()` calls VirtualiZarr internally.  The default parser is `DMRPPParser` (fastest when NASA DMR++ sidecars exist); earthaccess auto-falls back to `HDFParser` when sidecars are absent.

| Parameter | What it does | Required for |
|---|---|---|
| `concat_dim` | Dimension to concatenate granules along | Always when `len(granules) > 1` |
| `group` | HDF5/NetCDF4 group path | Multi-group files (e.g. TEMPO product / geolocation) |
| `pad="auto"` | Pad ragged arrays to union shape | TEMPO (variable scanlines per granule) |
| `mosaic_dims` | Union spatial tiles into common grid | ITS-LIVE (overlapping spatial tiles) |
| `loadable_variables` | Coordinates to materialise in memory | Mosaic dims must be loaded; time is useful for concat |

**TEMPO example** — ragged scanlines, no mosaicking:


In [ ]:
import time

t0 = time.perf_counter()
vds = earthaccess.virtualize(
    granules,
    access="indirect",          # HTTPS (works everywhere)
    group="product",            # TEMPO L2 group containing NO₂ variables
    concat_dim="time",
    pad="auto",                 # pad variable scanline counts to union shape
    loadable_variables=["time", "latitude", "longitude"],
    parallel="dask",           # will use the active Dask Client if any
)
elapsed = time.perf_counter() - t0

print(f"Virtualized in {elapsed:.2f} s")
print(f"Dataset dims: {dict(vds.dims)}")
print(f"Data variables: {list(vds.data_vars)}")
print(f"Coords: {list(vds.coords)}")

### 4a. Multi-group files (e.g. TEMPO)

TEMPO separates product variables (`NO2`, `CloudFraction`, …) from geolocation (`latitude`, `longitude`, `SolarZenithAngle`, …) into different HDF5 groups.  Each group must be virtualized separately and then merged with `xr.merge()`.

**`loadable_variables` per group**

- **`time`** — required for both groups. It is the 1-D coordinate on the scanline dimension; `xr.concat` needs its values to build the combined time index.
- **`latitude` / `longitude`** — optional. If loaded, they become concrete numpy arrays (useful for plotting / masking). If left virtual, they remain `ManifestArray`-backed and are read chunk-by-chunk during analysis.

In [ ]:
# Uncomment if you need both product and geolocation groups
# import xarray as xr
#
# product = earthaccess.virtualize(
#     granules,
#     group="product",
#     concat_dim="time",
#     pad="auto",
#     loadable_variables=["time"],  # time is required for concat
# )
# geolocation = earthaccess.virtualize(
#     granules,
#     group="geolocation",
#     concat_dim="time",
#     pad="auto",
#     loadable_variables=["time", "latitude", "longitude"],
# )
# vds = xr.merge([product, geolocation])
# print(f"Merged dims: {dict(vds.dims)}")

### 4b. Spatial mosaic (e.g. ITS-LIVE)

For tiled data that covers different geographic footprints, use `mosaic_dims` to union tiles into a common grid before concatenation.

In [ ]:
# Uncomment for ITS-LIVE style mosaic
# vds = earthaccess.virtualize(
#     granules,
#     concat_dim="time",
#     mosaic_dims=["y", "x"],
#     loadable_variables=["y", "x", "time"],
#     pad="none",
# )
# print(f"Mosaic dims: {dict(vds.dims)}")

## 5. Serialize to a persistent virtual store

Choose the backend that matches your access pattern:

| Backend | Format | Best for |
|---|---|---|
| **kerchunk JSON** | Single `.json` reference file | < 1 M chunks, easy sharing |
| **kerchunk parquet** | Columnar reference directory | > 1 M chunks, fast random lookup |
| **icechunk** | Transactional virtual-chunk store | Cloud-native, versioning, multi-writer |


In [ ]:
OUTPUT_PATH = "./stores/tempo_product.parquet"
OUTPUT_FORMAT = "parquet"   # "json" or "parquet"

t0 = time.perf_counter()
if OUTPUT_FORMAT == "json":
    vds.vz.to_kerchunk(OUTPUT_PATH, format="json")
else:
    vds.vz.to_kerchunk(OUTPUT_PATH, format="parquet")
print(f"Written in {time.perf_counter() - t0:.2f} s to {OUTPUT_PATH}")

### 5a. icechunk (optional)

icechunk stores virtual chunks in a transactional Zarr v3 repository.  It supports versioning and works well on cloud object storage.

In [ ]:
# Uncomment to write to icechunk instead
# import icechunk
#
# ICE_PATH = "./stores/tempo_product.icechunk"
# repo = icechunk.Repository.create(
#     icechunk.local_filesystem_storage(ICE_PATH)
# )
# session = repo.writable_session("main")
# vds.vz.to_icechunk(session.store)
# session.commit("virtualize_tempo_product")
# print(f"icechunk store written to {ICE_PATH}")

## 6. Re-open and compute

The output is a self-contained reference store.  Anyone with the store file(s) can lazily access the original NASA data without re-running the virtualization step.

**Note on Dask**: `xr.open_dataset(..., engine="kerchunk")` returns a dask-backed Dataset.  If a Dask Client is active, the `.compute()` or `.mean()` call below will use the cluster.

In [ ]:
import xarray as xr

# Re-open from kerchunk references
ds = xr.open_dataset(OUTPUT_PATH, engine="kerchunk")
print(f"Re-opened dims: {dict(ds.dims)}")
print(ds)

In [ ]:
# Compute a global mean (triggers chunk-level GETs from NASA S3/HTTPS)
t0 = time.perf_counter()
mean_no2 = ds["vertical_column"].mean()   # adjust variable name as needed
result = mean_no2.compute()
print(f"Mean computed in {time.perf_counter() - t0:.2f} s: {float(result):.4f}")

## 7. (Optional) Re-open from icechunk

If you wrote to icechunk, use the icechunk Python API to get a Zarr v3 store and pass it to xarray.

In [ ]:
# Uncomment to re-open from icechunk
# import icechunk
# repo = icechunk.Repository.open(icechunk.local_filesystem_storage(ICE_PATH))
# session = repo.readonly_session("main")
# ds_ice = xr.open_zarr(session.store, consolidated=False)
# print(f"icechunk dims: {dict(ds_ice.dims)}")

## Appendix: full parameter reference

```python
earthaccess.virtualize(
    granules,
    *,
    access="indirect",               # "indirect" (HTTPS) or "direct" (S3, requires creds)
    load=False,                      # True = materialise via kerchunk round-trip
    group="/",                      # HDF5/NetCDF4 group path
    concat_dim="time",              # Dimension to concatenate on
    mosaic_dims=None,               # Spatial dims to union (e.g. ["y","x"])
    pad=None,                       # "auto" or "none" or dict
    loadable_variables=None,        # Coords to materialise (required for mosaic_dims)
    preprocess=None,                # Callable[[xr.Dataset], xr.Dataset]
    data_vars="all",               # forwarded to xr.combine_nested
    coords="different",           # forwarded to xr.combine_nested
    compat="no_conflicts",         # forwarded to xr.combine_nested
    combine_attrs="drop_conflicts",  # forwarded to xr.combine_nested
    parallel="dask",               # "dask", "lithops", or False
    parser="DMRPPParser",         # "DMRPPParser" (default), "HDFParser", "NetCDF3Parser"
    reference_dir=None,            # Temporary dir for kerchunk refs when load=True
    reference_format="json",     # "json" or "parquet" when load=True
)
```